In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager
from mpl_toolkits.axes_grid1 import make_axes_locatable

arial_path = "/media/scratch/fy2306/tools/fonts"
font_files = font_manager.findSystemFonts(fontpaths=arial_path)

for file in font_files:
    font_manager.fontManager.addfont(file)
    
mpl.rcParams['font.family'] = 'Arial'
import seaborn as sns
import warnings
from matplotlib.patches import Patch
from statannotations.Annotator import Annotator
warnings.filterwarnings('ignore')

In [ ]:
def off_target_plot_final(
		date, 
		test_name, 
		baseline, 
		treatments, 
		day_numbers, 
		merge_method_replicate, per_time_unit=True):
	
	# merge 4 replicates
	# copied from handling negative control sgRNAs
	df_test = None
	for treatment in treatments:
		df_test_path = f"/media/scratch/fy2306/projects/base_editing/data/{date}/mageck/{test_name}/MYC_U1.{treatment}_vs_{baseline}.sgrna_summary.txt"
		df_test_rep = pd.read_csv(df_test_path, sep="\t", usecols=["sgrna", "Gene", "LFC"])
		df_test_rep = df_test_rep[df_test_rep["Gene"].isin(["MYC_GFP", "MYC_SNP1", "MYC_SNP2", "MYC_SNP3", "MYC_STOP"])]
		df_test_rep = df_test_rep.drop('Gene', axis=1)
		df_test_rep = df_test_rep.rename(columns={"LFC": f"LFC_{treatment}"})
		if df_test is None:
			df_test = df_test_rep
		else:
			# Merge on 'sgrna'
			df_test = pd.merge(df_test, df_test_rep, on="sgrna", how="inner")
	print(len(df_test))

	treatment_lfc_columns = [f"LFC_{treatment}" for treatment in treatments]

	if per_time_unit:
		for col, day in zip(treatment_lfc_columns, day_numbers):
			df_test[col] = df_test[col] / (day + 0) # per time unit. 

	if merge_method_replicate == "mean":
		df_test["per time unit LFC"] = df_test[treatment_lfc_columns].mean(axis=1)
	df_test = df_test[["sgrna", "per time unit LFC"]]

	df_off_path = "/media/scratch/fy2306/projects/base_editing/data/bowtie/all/all_sgrna_seqs.bowtie_hg38.processed.tsv"
	df_off = pd.read_csv(df_off_path, sep="\t")
	df_lib_path = "/media/scratch/fy2306/projects/base_editing/data/grna_type/MYC-lib-for-mageck.type.myc2_indel.organized.txt"
	df_lib = pd.read_csv(df_lib_path, sep="\t", usecols=["id", "seq_name", "indel_consequences", "indel_repeats"])

	df = pd.merge(df_lib, df_test, left_on='id', right_on='sgrna')
	df = pd.merge(df, df_off, left_on='id', right_on='id', how='left')

	df['Alignments_NM0'] = df['Alignments_NM0'].fillna(0).astype(int)
	df['Alignments_NM1'] = df['Alignments_NM1'].fillna(0).astype(int)
	df['Alignments_NM_less_than_1'] = df['Alignments_NM_less_than_1'].fillna(0).astype(int)	# actually no more than 1
	df[df['Alignments_NM_less_than_1'] == 0].groupby('indel_consequences').size().sort_values(ascending=False)
	# overlapping with GFP
	df.loc[(df['Alignments_NM_less_than_1'] == 0) & (df['indel_consequences'] == 'coding'), 'Alignments_NM_less_than_1'] = 1
	df.loc[(df['Alignments_NM_less_than_1'] == 0) & (df['indel_consequences'] == '3_prime_UTR'), 'Alignments_NM_less_than_1'] = 1


	# df_nc = df[df['seq_name'].isin(["AAVS1", "NT", "Random"])]
	# if use: add neg ctrl mismatch
	df = df[(df['seq_name'].isin(["MYC_GFP", "MYC_SNP1", "MYC_SNP2", "MYC_SNP3", "MYC_STOP"])) & (df['indel_consequences'] != "GFP")]

	# plot prep
	df['log10_alignments'] = np.log10(df['Alignments_NM_less_than_1'])
	print(df[df['log10_alignments'] == 0].groupby('indel_repeats').size())	# perfect match
	# Why are indel_repeats=True having perfect match?
	perfect_match_repeats = df[(df['log10_alignments'] == 0) & (df['indel_repeats'] == True)]['id'].tolist()
	print(perfect_match_repeats[:5])
	# they are simple repeats

	# figures for the text
	n1 = len(df[df['indel_repeats'] == False])
	n2 = len(df[(df['indel_repeats'] == False) & (df['log10_alignments'] == 0)])
	print(f"non-repeat-targeting sgRNAs: {n1}")
	print(f"non-repeat-targeting sgRNAs with no off-targets: {n2}")
	print(n2/n1)

	# scatter plot

	fig, ax = plt.subplots(figsize=(4.5, 2.7))
	divider = make_axes_locatable(ax)
	sns.scatterplot(data=df, x='log10_alignments', y='per time unit LFC', hue='indel_repeats', s=1.5, alpha=1.0, palette={False: '#FF0066', True: '#40007F'}, ax=ax)
	plt.xlabel('Log10(# of off-targets)', fontsize=13)
	plt.ylabel('sgRNA phenotype score', fontsize=13)
	plt.xticks(fontsize=12)
	plt.yticks(fontsize=12)
	plt.xlim(-0.1, 6.1)
	plt.grid(False)
	plt.gca().legend_.remove()
	plt.tight_layout()
	ax_box = ax.get_position()
	ax.spines['top'].set_visible(False)
	ax.spines['right'].set_visible(False)
	plt.savefig("/media/scratch/fy2306/projects/base_editing/plots/screening/offtargets_sgrna_lfc.pdf", 
				bbox_inches="tight",
				dpi=300,              
				transparent=True,
				format='pdf')
	plt.show()

	# sgRNA % plot
	plt.figure(figsize=(4.5, 2.5))

	ax = sns.histplot(
		data=df,
		x='log10_alignments',
		hue='indel_repeats',
		stat='percent',
		multiple='dodge',
		common_norm=False,
		bins=30,
		palette={False: '#FF0066', True: '#40007F'}
	)

	plt.xlabel('Log10(# of off-targets)', fontsize=13)
	plt.ylabel('sgRNA % (of each group)', fontsize=13)
	plt.xticks(fontsize=12)
	plt.yticks(fontsize=12)
	plt.xlim(-0.1, 6.1)
	plt.grid(False)
	sns.move_legend(ax, "upper right", title="Indel repeats", fontsize=12, title_fontsize=12, frameon=False)
	plt.tight_layout()
	ax.set_position(ax_box)
	ax.spines['top'].set_visible(False)
	ax.spines['right'].set_visible(False)
	plt.savefig("/media/scratch/fy2306/projects/base_editing/plots/screening/offtargets_percentage.pdf", 
				bbox_inches="tight",
				dpi=300,              
				transparent=True,
				format='pdf')
	plt.show()

In [ ]:
date = "20250416"
test_name = "test-1-standard/test"
baseline = "Cas9-HMOI_D0"
treatments = ["Cas9-LMOI_D20", "Cas9-LMOI_D8", "Cas9_SpRY_L_MOI_D_20", "Cas9_SpRY_L_MOI_D_8"]
day_numbers = [20, 8, 20, 8]
merge_method_replicate = "mean"
off_target_plot_final(date, test_name, baseline, treatments, day_numbers, merge_method_replicate, per_time_unit=True)